# v2 BRM Sensitivity Study

Studies the sensitivity of **Bellman Residual Minimization (BRM)** by comparing the
original **joint-update BRM** against the refined **separated-update BRM** in two settings:

1. **Frictionless benchmark** with an analytical solution.
2. **Frictional benchmark** with adjustment costs, using **PFI** as the benchmark.

The notebook uses a **fixed wall-clock budget** for all BRM variants. Reported models are
selected by the best external benchmark metric **within budget**, not by the last training step.


---
# Section 0: Setup


In [ ]:
import sys, os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib-codex")

import time
from dataclasses import replace

import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from scipy.interpolate import RegularGridInterpolator

sys.path.append(os.path.abspath(".."))

from src.v2.benchmarks import (
    load_manifest,
    load_method_bundle,
    load_named_dataset,
    load_pfi_bundle,
    prepare_benchmark_run,
    restore_selected_models,
    save_figure,
    save_manifest_sections,
    save_method_bundle,
    save_named_dataset,
    save_pfi_bundle,
    save_plot_inputs,
    save_summary_rows,
)
from src.v2.data.generator import DataGenerator, DataGeneratorConfig
from src.v2.environments.basic_investment import (
    BasicInvestmentEnv,
    EconomicParams,
    ShockParams,
    compute_frictionless_policy,
)
from src.v2.experimental import train_brm_joint
from src.v2.networks.policy import PolicyNetwork
from src.v2.networks.state_value import StateValueNetwork
from src.v2.solvers import solve_pfi, PFIConfig, GridConfig
from src.v2.trainers.brm import train_brm
from src.v2.trainers.config import BRMConfig, OptimizerConfig
from src.v2.trainers.core import evaluate_euler_residual
from src.v2.utils.annealing import AnnealingSchedule
from src.v2.utils.seeding import fold_in_seed, seed_runtime

print(f"TensorFlow {tf.__version__}")
print(f"GPU available: {len(tf.config.list_physical_devices('GPU')) > 0}")


In [ ]:
TRAINING_PROFILE = "FAST_DEBUG"  # "FAST_DEBUG" | "BALANCED"

SAVE_RUN = True
LOAD_RUN_DIR = None  # e.g. "outputs/v2_05_brm_sensitivity/latest"
RUN_TAG = None
SNAPSHOT_TARGETS = ("policy", "value_net")
STRICT_REPRODUCIBILITY = False

PROFILE_SETTINGS = {
    "FAST_DEBUG": {
        "data_n_paths": 64,
        "data_horizon": 32,
        "eval_horizon": 24,
        "eval_traj_samples": 64,
        "analytic_gap_samples": 1024,
        "pfi_gap_samples": 1024,
        "calibration_steps": 15,
        "calibration_interval": 5,
        "calibration_tail_points": 3,
        "target_eval_seconds": 5.0,
        "step_cap_buffer": 1.25,
        "frictionless_budget_sec": 90.0,
        "frictional_budget_sec": 150.0,
        "anneal_fraction": 0.50,
        "pfi_exo_size": 80,
        "pfi_endo_size": 80,
        "pfi_act_size": 800,
        "run_seed_sweep": False,
        "run_warm_sweep": False,
        "seed_values": [101, 202],
        "warm_sweep": [("cold", 0), ("weak", 50), ("standard", 200)],
    },
    "BALANCED": {
        "data_n_paths": 128,
        "data_horizon": 48,
        "eval_horizon": 32,
        "eval_traj_samples": 128,
        "analytic_gap_samples": 2048,
        "pfi_gap_samples": 2048,
        "calibration_steps": 20,
        "calibration_interval": 5,
        "calibration_tail_points": 4,
        "target_eval_seconds": 10.0,
        "step_cap_buffer": 1.35,
        "frictionless_budget_sec": 8 * 60.0,
        "frictional_budget_sec": 12 * 60.0,
        "anneal_fraction": 0.75,
        "pfi_exo_size": 120,
        "pfi_endo_size": 120,
        "pfi_act_size": 1600,
        "run_seed_sweep": True,
        "run_warm_sweep": True,
        "seed_values": [101, 202, 303, 404, 505],
        "warm_sweep": [("cold", 0), ("weak", 50), ("standard", 200), ("strong", 600)],
    },
}

if TRAINING_PROFILE not in PROFILE_SETTINGS:
    raise ValueError(f"Unknown TRAINING_PROFILE: {TRAINING_PROFILE}")

P = PROFILE_SETTINGS[TRAINING_PROFILE]
MASTER_SEED = (20, 26)
N_LAYERS = 2
N_NEURONS = 128
BATCH_SIZE = 256
BRM_WARM_STEPS = 200
ANNEAL_INIT_TEMP = 1.0
EVAL_TEMP = 1e-6
ANNEAL_BUFFER = 0.15
CALIBRATION_STEPS = P["calibration_steps"]
CALIBRATION_INTERVAL = P["calibration_interval"]
CALIBRATION_TAIL_POINTS = P["calibration_tail_points"]
TARGET_EVAL_SECONDS = P["target_eval_seconds"]
STEP_CAP_BUFFER = P["step_cap_buffer"]
FRICTIONLESS_BUDGET_SEC = P["frictionless_budget_sec"]
FRICTIONAL_BUDGET_SEC = P["frictional_budget_sec"]
FULL_ANNEAL_FRACTION = P["anneal_fraction"]
EVAL_HORIZON = P["eval_horizon"]
EVAL_TRAJ_SAMPLES = P["eval_traj_samples"]
ANALYTIC_GAP_SAMPLES = P["analytic_gap_samples"]
PFI_GAP_SAMPLES = P["pfi_gap_samples"]
RUN_SEED_SWEEP = P["run_seed_sweep"]
RUN_WARM_SWEEP = P["run_warm_sweep"]
SEED_SWEEP_VALUES = list(P["seed_values"])
WARM_SWEEP_LEVELS = list(P["warm_sweep"])

OPT = OptimizerConfig(learning_rate=1e-3, clipnorm=100.0)

FRICTIONLESS_VARIANTS = {
    "joint_cold": {
        "label": "Joint + Cold",
        "trainer": train_brm_joint,
        "warm_start_steps": 0,
        "color": "#C44E52",
    },
    "joint_warm": {
        "label": "Joint + Warm",
        "trainer": train_brm_joint,
        "warm_start_steps": BRM_WARM_STEPS,
        "color": "#DD8452",
    },
    "separated_cold": {
        "label": "Separated + Cold",
        "trainer": train_brm,
        "warm_start_steps": 0,
        "color": "#55A868",
    },
    "separated_warm": {
        "label": "Separated + Warm",
        "trainer": train_brm,
        "warm_start_steps": BRM_WARM_STEPS,
        "color": "#4C72B0",
    },
}
FRICTIONLESS_ORDER = list(FRICTIONLESS_VARIANTS.keys())
FRICTIONAL_VARIANTS = ["joint_warm", "separated_warm"]

RUN = prepare_benchmark_run(
    "v2_05_brm_sensitivity",
    save_run=SAVE_RUN,
    load_run_dir=LOAD_RUN_DIR,
    results_root="outputs",
    run_tag=RUN_TAG,
)

print(f"Profile: {TRAINING_PROFILE}")
print(f"Master seed: {MASTER_SEED}")
print(f"Snapshot targets: {SNAPSHOT_TARGETS}")
print(f"Strict reproducibility: {STRICT_REPRODUCIBILITY}")
print(f"Run mode: {RUN['mode']}")
if RUN["mode"] != "memory":
    print(f"Run directory: {RUN['run_dir']}")
print(f"Frictionless budget: {FRICTIONLESS_BUDGET_SEC/60:.1f} min/run")
print(f"Frictional budget:   {FRICTIONAL_BUDGET_SEC/60:.1f} min/run")
print(f"Seed sweep enabled: {RUN_SEED_SWEEP}")
print(f"Warm sweep enabled: {RUN_WARM_SWEEP}")


---
# Section 1: Environments, Data, and Shared Grids


In [ ]:
shock_params = ShockParams(mu=0.0, rho=0.7, sigma=0.15)

frictionless_params = EconomicParams(
    interest_rate=0.04,
    depreciation_rate=0.15,
    production_elasticity=0.7,
    cost_convex=0.0,
    cost_fixed=0.0,
)
full_params = EconomicParams(
    interest_rate=0.04,
    depreciation_rate=0.15,
    production_elasticity=0.7,
    cost_convex=0.2,
    cost_fixed=0.02,
)

env_frictionless_eval = BasicInvestmentEnv(
    econ_params=frictionless_params,
    shock_params=shock_params,
)
env_full_eval = BasicInvestmentEnv(
    econ_params=full_params,
    shock_params=shock_params,
)

print("Frictionless discount:", round(env_frictionless_eval.discount(), 6))
print("Full-model discount:", round(env_full_eval.discount(), 6))

data_cfg = DataGeneratorConfig(
    n_paths=P["data_n_paths"],
    horizon=P["data_horizon"],
    master_seed=MASTER_SEED,
)

if RUN["mode"] == "load":
    frictionless_train_flat = load_named_dataset(RUN, "frictionless_train_flat", fmt="flat")
    frictionless_val_flat = load_named_dataset(RUN, "frictionless_val_flat", fmt="flat")
    frictionless_val_traj = load_named_dataset(RUN, "frictionless_val_traj", fmt="trajectory")
    frictionless_accuracy_flat = load_named_dataset(RUN, "frictionless_accuracy_flat", fmt="flat")

    full_train_flat = load_named_dataset(RUN, "full_train_flat", fmt="flat")
    full_val_flat = load_named_dataset(RUN, "full_val_flat", fmt="flat")
    full_val_traj = load_named_dataset(RUN, "full_val_traj", fmt="trajectory")
    full_pfi_gap_flat = load_named_dataset(RUN, "full_pfi_gap_flat", fmt="flat")
else:
    gen_fric = DataGenerator(env_frictionless_eval, data_cfg)
    frictionless_train_flat = gen_fric.get_flattened_dataset("train")
    frictionless_val_flat = gen_fric.get_flattened_dataset("val")
    frictionless_val_traj = gen_fric.get_trajectory_dataset("val")
    frictionless_accuracy_flat = {k: v[:ANALYTIC_GAP_SAMPLES] for k, v in frictionless_val_flat.items()}

    gen_full = DataGenerator(env_full_eval, data_cfg)
    full_train_flat = gen_full.get_flattened_dataset("train")
    full_val_flat = gen_full.get_flattened_dataset("val")
    full_val_traj = gen_full.get_trajectory_dataset("val")
    full_pfi_gap_flat = {k: v[:PFI_GAP_SAMPLES] for k, v in full_val_flat.items()}

    save_named_dataset(RUN, "frictionless_train_flat", frictionless_train_flat, fmt="flat")
    save_named_dataset(RUN, "frictionless_val_flat", frictionless_val_flat, fmt="flat")
    save_named_dataset(RUN, "frictionless_val_traj", frictionless_val_traj, fmt="trajectory")
    save_named_dataset(RUN, "frictionless_accuracy_flat", frictionless_accuracy_flat, fmt="flat")

    save_named_dataset(RUN, "full_train_flat", full_train_flat, fmt="flat")
    save_named_dataset(RUN, "full_val_flat", full_val_flat, fmt="flat")
    save_named_dataset(RUN, "full_val_traj", full_val_traj, fmt="trajectory")
    save_named_dataset(RUN, "full_pfi_gap_flat", full_pfi_gap_flat, fmt="flat")

n_grid = 400 if TRAINING_PROFILE == "FAST_DEBUG" else 500
z_grid = np.linspace(float(env_frictionless_eval.z_min), float(env_frictionless_eval.z_max), n_grid)
k_grid = np.linspace(float(env_frictionless_eval.k_min), float(env_frictionless_eval.k_max), n_grid)
k_star = float(env_frictionless_eval.k_star)
z_mean = float(np.exp(shock_params.mu))

save_plot_inputs(
    RUN,
    "slice_grids",
    {
        "z_grid": z_grid,
        "k_grid": k_grid,
        "k_star": [k_star],
        "z_mean": [z_mean],
    },
)

save_manifest_sections(
    RUN,
    notebook="v2_05_brm_sensitivity",
    training_profile=TRAINING_PROFILE,
    snapshot_targets=list(SNAPSHOT_TARGETS),
    strict_reproducibility=STRICT_REPRODUCIBILITY,
    data_config=data_cfg,
    frictionless_params=frictionless_params,
    full_params=full_params,
    shock_params=shock_params,
)

print("Datasets ready.")
print("  frictionless train flat:", frictionless_train_flat["s_endo"].shape[0])
print("  full train flat:       ", full_train_flat["s_endo"].shape[0])


---
# Section 2: Helpers


In [ ]:
def make_full_training_env(decay_rate=None):
    env = BasicInvestmentEnv(econ_params=full_params, shock_params=shock_params)
    if decay_rate is not None:
        def schedule_factory(_decay=decay_rate):
            return AnnealingSchedule(
                init_temp=ANNEAL_INIT_TEMP,
                min_temp=EVAL_TEMP,
                decay_rate=_decay,
                buffer=ANNEAL_BUFFER,
            )
        env.annealing_schedule = schedule_factory
    return env


def derived_seed(*tokens):
    return fold_in_seed(MASTER_SEED, *tokens)


def seed_experiment(*tokens):
    return seed_runtime(
        MASTER_SEED,
        *tokens,
        strict_reproducibility=STRICT_REPRODUCIBILITY,
    )


def make_policy(env, name="policy", seed=None):
    net = PolicyNetwork(
        state_dim=env.state_dim(),
        action_dim=env.action_dim(),
        **env.action_spec(),
        n_layers=N_LAYERS,
        n_neurons=N_NEURONS,
        seed=seed,
        name=name,
    )
    net(tf.zeros((1, env.state_dim())))
    return net


def make_value_net(env, name="value_net", seed=None):
    net = StateValueNetwork(
        state_dim=env.state_dim(),
        n_layers=N_LAYERS,
        n_neurons=N_NEURONS,
        seed=seed,
        name=name,
    )
    net(tf.zeros((1, env.state_dim())))
    return net


def sanitize_config(base_config, **updates):
    payload = dict(
        monitor=None,
        mode="min",
        threshold=None,
        stop_patience=1,
        stop_min_delta=0.0,
        stop_rel_delta=0.0,
        min_steps_before_stop=0,
        max_wall_time_sec=None,
        weight_history=None,
        checkpoint_history=None,
        snapshot_targets=SNAPSHOT_TARGETS,
    )
    payload.update(updates)
    return replace(base_config, **payload)


def estimate_steady_step_cost(history, wall_time_sec):
    steps = np.asarray(history["step"], dtype=np.float64)
    elapsed = np.asarray(history["elapsed_sec"], dtype=np.float64)
    if steps.size >= 2:
        tail = min(CALIBRATION_TAIL_POINTS, steps.size)
        steps_tail = steps[-tail:]
        elapsed_tail = elapsed[-tail:]
        step_span = steps_tail[-1] - steps_tail[0]
        elapsed_span = elapsed_tail[-1] - elapsed_tail[0]
        if step_span > 0 and elapsed_span > 0:
            return elapsed_span / step_span
    total_steps = max(int(steps[-1]) + 1 if steps.size else 1, 1)
    return wall_time_sec / total_steps


def calibrate_time_per_step(env_factory, label, train_fn, make_nets_fn, datasets, base_config):
    env_cal = env_factory()
    seed_experiment("calibration", label)
    measure_config = sanitize_config(
        base_config,
        n_steps=CALIBRATION_STEPS,
        eval_interval=CALIBRATION_INTERVAL,
    )
    nets = make_nets_fn(env_cal)
    measured = train_fn(env_cal, *nets, *datasets, config=measure_config)
    t_per_step = estimate_steady_step_cost(measured["history"], measured["wall_time_sec"])
    eval_interval = max(int(round(TARGET_EVAL_SECONDS / max(t_per_step, 1e-8))), 1)
    n_step_cap = max(int(np.ceil(base_config.max_wall_time_sec / max(t_per_step, 1e-8) * STEP_CAP_BUFFER)), 1)
    return {
        "t_per_step": t_per_step,
        "eval_interval": eval_interval,
        "n_step_cap": n_step_cap,
    }


def derive_decay_rate(target_n_anneal, init_temp=ANNEAL_INIT_TEMP, min_temp=EVAL_TEMP, buffer=ANNEAL_BUFFER):
    target_n_anneal = max(int(target_n_anneal), 1)
    effective_n = max(target_n_anneal / (1.0 + buffer), 1.0)
    return float(np.exp((np.log(min_temp) - np.log(init_temp)) / effective_n))


def persist_figure(fig, filename):
    save_figure(RUN, fig, filename)


def persist_method(bundle_name, entry):
    extra = {
        "label": entry.get("label", bundle_name),
        "section": entry.get("section"),
        "calibration": entry.get("calibration"),
        "snapshot_targets": list(entry.get("snapshot_targets", SNAPSHOT_TARGETS)),
        "budget_sec": entry.get("budget_sec"),
        "decay_rate": entry.get("decay_rate"),
        "n_anneal": entry.get("n_anneal"),
    }
    save_method_bundle(
        RUN,
        bundle_name,
        result=entry.get("result"),
        checkpoint_history=entry.get("checkpoint_history"),
        checkpoint_metrics=entry.get("checkpoint_metrics"),
        selected=entry.get("selected"),
        extra_sections=extra,
    )


def restore_entry_at_step(entry, step):
    models = {"policy": entry["policy"]}
    if entry.get("value_net") is not None:
        models["value_net"] = entry["value_net"]
    restore_selected_models(models, entry["checkpoint_history"], step)


def load_saved_method(bundle_name, env):
    entry = load_method_bundle(RUN, bundle_name)
    checkpoint_history = entry.get("checkpoint_history") or []
    policy = make_policy(env, name=f"policy_{bundle_name}")
    has_value = any("value_net" in item.get("models", {}) for item in checkpoint_history)
    value_net = make_value_net(env, name=f"value_{bundle_name}") if has_value else None
    if entry.get("selected") is not None and checkpoint_history:
        restore_entry_at_step({**entry, "policy": policy, "value_net": value_net}, entry["selected"]["selected_step"])
    elif checkpoint_history:
        restore_entry_at_step({**entry, "policy": policy, "value_net": value_net}, checkpoint_history[-1]["step"])
    entry["policy"] = policy
    entry["value_net"] = value_net
    entry["label"] = entry["result"].get("label", bundle_name)
    entry["section"] = entry["result"].get("section")
    entry["calibration"] = entry["result"].get("calibration")
    entry["budget_sec"] = entry["result"].get("budget_sec")
    entry["decay_rate"] = entry["result"].get("decay_rate")
    entry["n_anneal"] = entry["result"].get("n_anneal")
    entry["snapshot_targets"] = tuple(entry["result"].get("snapshot_targets", SNAPSHOT_TARGETS))
    entry["checkpoint_index"] = {
        int(step): idx for idx, step in enumerate(entry["checkpoint_metrics"]["step"])
    }
    return entry


def extract_checkpoint_metrics(result, keys):
    history = result["history"]
    return {key: list(history[key]) for key in keys if key in history}


def select_best_checkpoint(bundle_name, entry, metric_key, tie_break_key="implied_value"):
    metrics = entry["checkpoint_metrics"]
    metric = np.asarray(metrics[metric_key], dtype=np.float64)
    tie = np.asarray(metrics.get(tie_break_key, np.zeros_like(metric)), dtype=np.float64)
    order = np.lexsort((-tie, metric))
    idx = int(order[0])
    selected = {
        "bundle_name": bundle_name,
        "selected_idx": idx,
        "selected_step": int(metrics["step"][idx]),
        "selected_time": float(metrics["elapsed_sec"][idx]),
        "selected_metric_name": metric_key,
        "selected_metric": float(metric[idx]),
        "selected_value": float(metrics.get("implied_value", [np.nan] * len(metric))[idx]),
        "selected_loss_br": float(metrics.get("loss_br", [np.nan] * len(metric))[idx]),
        "selected_loss_foc": float(metrics.get("loss_foc", [np.nan] * len(metric))[idx]),
    }
    if tie_break_key in metrics:
        selected[f"selected_{tie_break_key}"] = float(metrics[tie_break_key][idx])
    entry["checkpoint_index"] = {int(step): j for j, step in enumerate(metrics["step"])}
    return selected


def add_selected_marker(ax, entry, metric_key):
    if entry.get("selected") is None:
        return
    idx = entry["checkpoint_index"][int(entry["selected"]["selected_step"])]
    x = entry["checkpoint_metrics"]["elapsed_sec"][idx]
    y = entry["checkpoint_metrics"][metric_key][idx]
    ax.scatter([x], [y], s=50, marker="o", edgecolor="black", facecolor="white", linewidth=1.2, zorder=6)


def analytic_kprime(z):
    return np.clip(
        compute_frictionless_policy(z, frictionless_params, shock_params),
        env_frictionless_eval.k_min,
        env_frictionless_eval.k_max,
    )


class AnalyticalPolicy:
    def __call__(self, s, training=False):
        del training
        s_np = s.numpy() if tf.is_tensor(s) else np.asarray(s, dtype=np.float32)
        s_np = np.asarray(s_np, dtype=np.float32)
        if s_np.ndim == 1:
            s_np = s_np[None, :]
        k = s_np[:, 0]
        z = s_np[:, 1]
        kprime = analytic_kprime(z)
        invest = np.clip(
            kprime - (1.0 - frictionless_params.depreciation_rate) * k,
            env_frictionless_eval.I_min,
            env_frictionless_eval.I_max,
        )
        return tf.convert_to_tensor(invest.reshape(-1, 1), dtype=tf.float32)


def build_pfi_policy(result, env):
    exo_1d = result["grids"]["exo_grids_1d"][0]
    endo_1d = result["grids"]["endo_grids_1d"][0]
    action_grid = result["policy_action"][:, :, 0].numpy()
    kprime_grid = result["policy_endo"][:, :, 0].numpy()
    action_interp = RegularGridInterpolator(
        (exo_1d, endo_1d),
        action_grid,
        method="linear",
        bounds_error=False,
        fill_value=None,
    )
    kprime_interp = RegularGridInterpolator(
        (exo_1d, endo_1d),
        kprime_grid,
        method="linear",
        bounds_error=False,
        fill_value=None,
    )

    class _PFIPolicy:
        def __call__(self, s, training=False):
            del training
            s_np = s.numpy() if tf.is_tensor(s) else np.asarray(s, dtype=np.float32)
            s_np = np.asarray(s_np, dtype=np.float32)
            if s_np.ndim == 1:
                s_np = s_np[None, :]
            pts = np.column_stack([s_np[:, 1], s_np[:, 0]])
            a = action_interp(pts).reshape(-1, env.action_dim())
            return tf.convert_to_tensor(a, dtype=tf.float32)

    return _PFIPolicy(), kprime_interp


def evaluate_implied_value(env, policy, traj_dataset, horizon=EVAL_HORIZON, n_samples=EVAL_TRAJ_SAMPLES,
                           temperature=EVAL_TEMP, gate_mode="soft"):
    gamma = env.discount()
    s_endo_0 = traj_dataset["s_endo_0"][:n_samples]
    z_path = traj_dataset["z_path"][:n_samples]
    T = min(horizon, z_path.shape[1] - 1)

    k = s_endo_0
    total_reward = tf.zeros(tf.shape(k)[0])
    discount_t = 1.0
    for t in range(T):
        z_t = z_path[:, t, :]
        s_t = env.merge_state(k, z_t)
        a_t = policy(s_t, training=False)
        r_t = tf.reshape(env.reward(s_t, a_t, temperature=temperature, gate_mode=gate_mode), [-1])
        total_reward = total_reward + discount_t * r_t
        k = env.endogenous_transition(k, a_t, z_t)
        discount_t = discount_t * gamma
    return float(tf.reduce_mean(total_reward))


def evaluate_analytic_policy_mae(env, policy, flat_dataset):
    s = env.merge_state(flat_dataset["s_endo"], flat_dataset["z"])
    a = policy(s, training=False)
    k_next = env.endogenous_transition(flat_dataset["s_endo"], a, flat_dataset["z"])
    target = analytic_kprime(flat_dataset["z"].numpy().reshape(-1))
    err = k_next.numpy().reshape(-1) - target.reshape(-1)
    return float(np.mean(np.abs(err)))


def evaluate_benchmark_policy_mae(env, policy, flat_dataset, benchmark_kprime_interp):
    s = env.merge_state(flat_dataset["s_endo"], flat_dataset["z"])
    a = policy(s, training=False)
    k_next = env.endogenous_transition(flat_dataset["s_endo"], a, flat_dataset["z"])
    points = np.column_stack([
        flat_dataset["z"].numpy().reshape(-1),
        flat_dataset["s_endo"].numpy().reshape(-1),
    ])
    target = benchmark_kprime_interp(points)
    err = k_next.numpy().reshape(-1) - np.asarray(target).reshape(-1)
    return float(np.mean(np.abs(err)))


def make_frictionless_eval_callback(eval_env, eval_flat, eval_traj, analytic_dataset):
    def _callback(step, _env, policy, value_net, val_dataset, train_temperature):
        del step, _env, value_net, val_dataset, train_temperature
        return {
            "analytic_policy_mae": evaluate_analytic_policy_mae(eval_env, policy, analytic_dataset),
            "implied_value": evaluate_implied_value(eval_env, policy, eval_traj),
            "euler_residual": evaluate_euler_residual(eval_env, policy, eval_flat),
        }
    return _callback


def make_frictional_eval_callback(eval_env, eval_flat, eval_traj, benchmark_kprime_interp, benchmark_value):
    def _callback(step, _env, policy, value_net, val_dataset, train_temperature):
        del step, _env, value_net, val_dataset, train_temperature
        implied_value = evaluate_implied_value(
            eval_env,
            policy,
            eval_traj,
            temperature=EVAL_TEMP,
            gate_mode="hard",
        )
        benchmark_policy_mae = evaluate_benchmark_policy_mae(
            eval_env,
            policy,
            eval_flat,
            benchmark_kprime_interp,
        )
        return {
            "implied_value": implied_value,
            "implied_value_gap_to_pfi": abs(implied_value - benchmark_value),
            "benchmark_policy_mae": benchmark_policy_mae,
        }
    return _callback


def get_policy_kprime(policy, env, states):
    states = tf.convert_to_tensor(states, dtype=tf.float32)
    k = states[:, 0:1]
    z = states[:, 1:2]
    a = policy(states, training=False)
    kp = env.endogenous_transition(k, a, z)
    return np.asarray(kp).reshape(-1)


---
# Section 3: Main Experiment A — Frictionless 2×2 Ablation

All four BRM variants share:
- the same frictionless environment
- the same offline dataset
- the same policy/value initialization
- the same wall-clock budget

The only changes are the **update structure** (joint vs separated) and the **critic warm start** (cold vs warm).


In [ ]:
frictionless_base = BRMConfig(
    batch_size=BATCH_SIZE,
    br_scale=1.0 / env_frictionless_eval.compute_reward_scale(),
    warm_start_steps=0,
    master_seed=MASTER_SEED,
    strict_reproducibility=STRICT_REPRODUCIBILITY,
    policy_optimizer=OPT,
    critic_optimizer=OPT,
    max_wall_time_sec=FRICTIONLESS_BUDGET_SEC,
    snapshot_targets=SNAPSHOT_TARGETS,
)
frictionless_eval_callback = make_frictionless_eval_callback(
    env_frictionless_eval,
    frictionless_val_flat,
    frictionless_val_traj,
    frictionless_accuracy_flat,
)

print(f"Calibrating frictionless BRM variants ({CALIBRATION_STEPS}-step probe)...")
if RUN["mode"] == "load":
    manifest = load_manifest(RUN)
    frictionless_cal = manifest["frictionless_calibration"]
    print("Loaded frictionless calibration from saved run.")
else:
    frictionless_cal = {}
    for name in FRICTIONLESS_ORDER:
        spec = FRICTIONLESS_VARIANTS[name]
        base = sanitize_config(
            frictionless_base,
            warm_start_steps=spec["warm_start_steps"],
        )
        cal = calibrate_time_per_step(
            lambda: BasicInvestmentEnv(econ_params=frictionless_params, shock_params=shock_params),
            f"frictionless_{name}",
            spec["trainer"],
            lambda env: (
                make_policy(env, name=f"cal_policy_{name}", seed=derived_seed("calibration", "frictionless", name, "policy")),
                make_value_net(env, name=f"cal_value_{name}", seed=derived_seed("calibration", "frictionless", name, "value")),
            ),
            (frictionless_train_flat, None),
            base,
        )
        frictionless_cal[name] = cal
    save_manifest_sections(RUN, frictionless_calibration=frictionless_cal)

print(f"{'Variant':>18s} | {'ms/step':>9s} | {'eval_int':>8s} | {'step_cap':>9s}")
print("-" * 56)
for name in FRICTIONLESS_ORDER:
    cal = frictionless_cal[name]
    print(f"{FRICTIONLESS_VARIANTS[name]['label']:>18s} | {cal['t_per_step']*1000:8.1f} | {cal['eval_interval']:8d} | {cal['n_step_cap']:9d}")


In [ ]:
frictionless_results = {}
main_policy_seed = derived_seed("frictionless", "main", "policy")
main_value_seed = derived_seed("frictionless", "main", "value")

for name in FRICTIONLESS_ORDER:
    bundle_name = f"frictionless_{name}"
    spec = FRICTIONLESS_VARIANTS[name]
    if RUN["mode"] == "load":
        entry = load_saved_method(bundle_name, env_frictionless_eval)
    else:
        seed_experiment("frictionless", "main", name)
        checkpoint_history = []
        policy = make_policy(env_frictionless_eval, name=f"policy_{bundle_name}", seed=main_policy_seed)
        value_net = make_value_net(env_frictionless_eval, name=f"value_{bundle_name}", seed=main_value_seed)
        config = sanitize_config(
            frictionless_base,
            n_steps=frictionless_cal[name]["n_step_cap"],
            eval_interval=frictionless_cal[name]["eval_interval"],
            max_wall_time_sec=FRICTIONLESS_BUDGET_SEC,
            checkpoint_history=checkpoint_history,
            snapshot_targets=SNAPSHOT_TARGETS,
            warm_start_steps=spec["warm_start_steps"],
        )
        result = spec["trainer"](
            env_frictionless_eval,
            policy,
            value_net,
            frictionless_train_flat,
            config=config,
            eval_callback=frictionless_eval_callback,
        )
        entry = {
            "label": spec["label"],
            "section": "frictionless_main",
            "policy": result["policy"],
            "value_net": result["value_net"],
            "result": result,
            "checkpoint_history": checkpoint_history,
            "checkpoint_metrics": extract_checkpoint_metrics(
                result,
                [
                    "step", "elapsed_sec", "train_temperature",
                    "loss", "loss_br", "loss_foc",
                    "analytic_policy_mae", "implied_value", "euler_residual",
                ],
            ),
            "calibration": frictionless_cal[name],
            "budget_sec": FRICTIONLESS_BUDGET_SEC,
            "snapshot_targets": SNAPSHOT_TARGETS,
        }
        entry["selected"] = select_best_checkpoint(bundle_name, entry, "analytic_policy_mae")
        persist_method(bundle_name, entry)
    entry["color"] = spec["color"]
    entry["name"] = name
    frictionless_results[name] = entry

frictionless_rows = []
for name in FRICTIONLESS_ORDER:
    entry = frictionless_results[name]
    sel = entry["selected"]
    idx = entry["checkpoint_index"][sel["selected_step"]]
    metrics = entry["checkpoint_metrics"]
    row = {
        "variant": entry["label"],
        "selected_step": int(sel["selected_step"]),
        "selected_time_sec": float(sel["selected_time"]),
        "analytic_policy_mae": float(metrics["analytic_policy_mae"][idx]),
        "implied_value": float(metrics["implied_value"][idx]),
        "euler_residual": float(metrics["euler_residual"][idx]),
        "loss_br": float(metrics["loss_br"][idx]),
        "loss_foc": float(metrics["loss_foc"][idx]),
        "wall_time_sec": float(entry["result"]["wall_time_sec"]),
    }
    frictionless_rows.append(row)

save_summary_rows(RUN, frictionless_rows, filename="frictionless_summary.csv")
print(f"{'Variant':>18s} | {'step':>6s} | {'time':>8s} | {'MAE':>10s} | {'Value':>12s} | {'Euler':>10s}")
print("-" * 78)
for row in frictionless_rows:
    print(
        f"{row['variant']:>18s} | {row['selected_step']:6d} | {row['selected_time_sec']:8.1f} | "
        f"{row['analytic_policy_mae']:10.4f} | {row['implied_value']:12.4f} | {row['euler_residual']:10.4f}"
    )


In [ ]:
analytic_policy = AnalyticalPolicy()
analytic_value = evaluate_implied_value(env_frictionless_eval, analytic_policy, frictionless_val_traj)
analytic_kprime_z = analytic_kprime(z_grid)
analytic_kprime_k = np.full_like(k_grid, analytic_kprime(np.array([z_mean]))[0])

s_at_kstar = np.column_stack([np.full_like(z_grid, k_star), z_grid]).astype(np.float32)
s_at_zmean = np.column_stack([k_grid, np.full_like(k_grid, z_mean)]).astype(np.float32)

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
metric_panels = [
    ("analytic_policy_mae", "Analytical Policy MAE", None),
    ("implied_value", "Implied Value", analytic_value),
    ("loss_br", "Bellman Residual Loss", None),
    ("loss_foc", "FOC Loss", None),
]
for ax, (metric_key, title, ref_line) in zip(axes.ravel(), metric_panels):
    for name in FRICTIONLESS_ORDER:
        entry = frictionless_results[name]
        m = entry["checkpoint_metrics"]
        ax.plot(m["elapsed_sec"], m[metric_key], color=entry["color"], lw=1.6, label=entry["label"])
        add_selected_marker(ax, entry, metric_key)
    if ref_line is not None:
        ax.axhline(ref_line, color="black", lw=1.2, ls="--", alpha=0.8)
    ax.set_title(title, fontweight="bold")
    ax.set_xlabel("Elapsed seconds")
    ax.grid(True, alpha=0.3)
handles = [Line2D([0], [0], color=frictionless_results[name]["color"], lw=2, label=frictionless_results[name]["label"]) for name in FRICTIONLESS_ORDER]
fig.legend(handles=handles, loc="lower center", bbox_to_anchor=(0.5, -0.03), ncol=2, frameon=False)
fig.suptitle("Frictionless BRM Ablation — External and Internal Metrics", fontweight="bold", fontsize=13)
plt.tight_layout(rect=[0, 0.06, 1, 0.96])
persist_figure(fig, "frictionless_convergence.png")
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].plot(z_grid, analytic_kprime_z, color="black", lw=2, ls="--", label="Analytical")
for name in FRICTIONLESS_ORDER:
    entry = frictionless_results[name]
    restore_entry_at_step(entry, entry["selected"]["selected_step"])
    kp = get_policy_kprime(entry["policy"], env_frictionless_eval, s_at_kstar)
    axes[0].plot(z_grid, kp, color=entry["color"], lw=1.6, label=entry["label"])
axes[0].set_title("k'(z) at k = k*", fontweight="bold")
axes[0].set_xlabel("Productivity z")
axes[0].set_ylabel("Next capital k'")
axes[0].grid(True, alpha=0.3)

axes[1].plot(k_grid, analytic_kprime_k, color="black", lw=2, ls="--", label="Analytical")
for name in FRICTIONLESS_ORDER:
    entry = frictionless_results[name]
    kp = get_policy_kprime(entry["policy"], env_frictionless_eval, s_at_zmean)
    axes[1].plot(k_grid, kp, color=entry["color"], lw=1.6, label=entry["label"])
axes[1].plot(k_grid, k_grid, color="gray", lw=1.0, ls=":", alpha=0.6)
axes[1].set_title("k'(k) at z = E[z]", fontweight="bold")
axes[1].set_xlabel("Current capital k")
axes[1].set_ylabel("Next capital k'")
axes[1].grid(True, alpha=0.3)

fig.legend(handles=handles + [Line2D([0], [0], color="black", lw=2, ls="--", label="Analytical")],
           loc="lower center", bbox_to_anchor=(0.5, -0.02), ncol=3, frameon=False)
fig.suptitle("Frictionless Selected Checkpoints — Policy Slice Validation", fontweight="bold", fontsize=13)
plt.tight_layout(rect=[0, 0.08, 1, 0.95])
persist_figure(fig, "frictionless_slices.png")
plt.show()


---
# Section 4: Main Experiment B — Internal vs External Diagnostics

These plots ask whether **low internal BRM losses** imply a **good policy**.
The key comparison is between the internal objectives (`loss_br`, `loss_foc`) and the external
accuracy metric (`analytic_policy_mae`).


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.5))
for name in ["joint_warm", "separated_warm"]:
    entry = frictionless_results[name]
    m = entry["checkpoint_metrics"]
    axes[0].scatter(m["loss_foc"], m["analytic_policy_mae"], s=28, alpha=0.75, color=entry["color"], label=entry["label"])
    axes[1].scatter(m["loss_br"], m["analytic_policy_mae"], s=28, alpha=0.75, color=entry["color"], label=entry["label"])
axes[0].set_title("FOC Loss vs Analytical Policy MAE", fontweight="bold")
axes[1].set_title("Bellman Residual Loss vs Analytical Policy MAE", fontweight="bold")
for ax in axes:
    ax.set_xlabel("Internal loss")
    ax.set_ylabel("Analytical Policy MAE")
    ax.grid(True, alpha=0.3)
fig.legend(loc="lower center", bbox_to_anchor=(0.5, -0.02), ncol=2, frameon=False)
plt.tight_layout(rect=[0, 0.08, 1, 1])
persist_figure(fig, "frictionless_internal_vs_external.png")
plt.show()


---
# Section 5: Main Experiment C — Frictional Benchmark with PFI

This section keeps the BRM comparison **BRM-focused**:
- **Joint + Warm**
- **Separated + Warm**

The benchmark is a **hard-indicator PFI** solution. Training still uses the smooth gate with annealing,
but evaluation is always on the hard benchmark through:
- `benchmark_policy_mae`
- `implied_value`


In [ ]:
full_base = BRMConfig(
    batch_size=BATCH_SIZE,
    br_scale=1.0 / env_full_eval.compute_reward_scale(),
    warm_start_steps=BRM_WARM_STEPS,
    master_seed=MASTER_SEED,
    strict_reproducibility=STRICT_REPRODUCIBILITY,
    policy_optimizer=OPT,
    critic_optimizer=OPT,
    max_wall_time_sec=FRICTIONAL_BUDGET_SEC,
    snapshot_targets=SNAPSHOT_TARGETS,
)

pfi_cfg = PFIConfig(
    max_iter=200,
    eval_steps=800,
    grid=GridConfig(
        exo_size=P["pfi_exo_size"],
        endo_size=P["pfi_endo_size"],
        action_size=P["pfi_act_size"],
    ),
)

if RUN["mode"] == "load":
    pfi_bundle = load_pfi_bundle(RUN)
    pfi_result = pfi_bundle["result"]
    pfi_summary = pfi_bundle.get("summary", {})
else:
    pfi_result = solve_pfi(env_full_eval, config=pfi_cfg)
    pfi_summary = {
        "converged": bool(pfi_result.get("converged", False)),
        "n_iter": int(pfi_result.get("n_iter", -1)),
        "solve_sec": float(pfi_result.get("wall_time_sec", np.nan)),
        "grid": {
            "exo_size": P["pfi_exo_size"],
            "endo_size": P["pfi_endo_size"],
            "action_size": P["pfi_act_size"],
        },
    }
    save_pfi_bundle(RUN, pfi_result, summary=pfi_summary)

pfi_policy, pfi_kprime_interp = build_pfi_policy(pfi_result, env_full_eval)
pfi_value = evaluate_implied_value(
    env_full_eval,
    pfi_policy,
    full_val_traj,
    temperature=EVAL_TEMP,
    gate_mode="hard",
)
full_eval_callback = make_frictional_eval_callback(
    env_full_eval,
    full_pfi_gap_flat,
    full_val_traj,
    pfi_kprime_interp,
    pfi_value,
)

print("PFI summary:")
for k, v in pfi_summary.items():
    print(f"  {k}: {v}")
print(f"  implied_value: {pfi_value:.6f}")

print(f"\nCalibrating frictional BRM variants ({CALIBRATION_STEPS}-step probe)...")
if RUN["mode"] == "load":
    manifest = load_manifest(RUN)
    full_cal = manifest["frictional_calibration"]
    print("Loaded frictional calibration from saved run.")
else:
    full_cal = {}
    for name in FRICTIONAL_VARIANTS:
        spec = FRICTIONLESS_VARIANTS[name]
        base = sanitize_config(full_base, warm_start_steps=spec["warm_start_steps"])
        cal = calibrate_time_per_step(
            lambda: make_full_training_env(),
            f"full_{name}",
            spec["trainer"],
            lambda env: (
                make_policy(env, name=f"cal_full_policy_{name}", seed=derived_seed("calibration", "full", name, "policy")),
                make_value_net(env, name=f"cal_full_value_{name}", seed=derived_seed("calibration", "full", name, "value")),
            ),
            (full_train_flat, None),
            base,
        )
        target_n_anneal = max(int(np.floor(FULL_ANNEAL_FRACTION * cal["n_step_cap"])), 1)
        cal["derived_decay"] = derive_decay_rate(target_n_anneal)
        schedule_preview = AnnealingSchedule(
            init_temp=ANNEAL_INIT_TEMP,
            min_temp=EVAL_TEMP,
            decay_rate=cal["derived_decay"],
            buffer=ANNEAL_BUFFER,
        )
        cal["n_anneal"] = int(schedule_preview.n_anneal)
        full_cal[name] = cal
    save_manifest_sections(RUN, frictional_calibration=full_cal)

print(f"{'Variant':>18s} | {'ms/step':>9s} | {'eval_int':>8s} | {'step_cap':>9s} | {'n_anneal':>9s}")
print("-" * 72)
for name in FRICTIONAL_VARIANTS:
    cal = full_cal[name]
    print(
        f"{FRICTIONLESS_VARIANTS[name]['label']:>18s} | {cal['t_per_step']*1000:8.1f} | "
        f"{cal['eval_interval']:8d} | {cal['n_step_cap']:9d} | {cal['n_anneal']:9d}"
    )


In [ ]:
full_results = {}
full_policy_seed = derived_seed("full", "main", "policy")
full_value_seed = derived_seed("full", "main", "value")

for name in FRICTIONAL_VARIANTS:
    bundle_name = f"full_{name}"
    spec = FRICTIONLESS_VARIANTS[name]
    if RUN["mode"] == "load":
        entry = load_saved_method(bundle_name, env_full_eval)
    else:
        seed_experiment("full", "main", name)
        checkpoint_history = []
        env_train = make_full_training_env(decay_rate=full_cal[name]["derived_decay"])
        policy = make_policy(env_train, name=f"policy_{bundle_name}", seed=full_policy_seed)
        value_net = make_value_net(env_train, name=f"value_{bundle_name}", seed=full_value_seed)
        config = sanitize_config(
            full_base,
            n_steps=full_cal[name]["n_step_cap"],
            eval_interval=full_cal[name]["eval_interval"],
            max_wall_time_sec=FRICTIONAL_BUDGET_SEC,
            checkpoint_history=checkpoint_history,
            snapshot_targets=SNAPSHOT_TARGETS,
            warm_start_steps=spec["warm_start_steps"],
        )
        result = spec["trainer"](
            env_train,
            policy,
            value_net,
            full_train_flat,
            config=config,
            eval_callback=full_eval_callback,
        )
        entry = {
            "label": spec["label"],
            "section": "full_main",
            "policy": result["policy"],
            "value_net": result["value_net"],
            "result": result,
            "checkpoint_history": checkpoint_history,
            "checkpoint_metrics": extract_checkpoint_metrics(
                result,
                [
                    "step", "elapsed_sec", "train_temperature",
                    "loss", "loss_br", "loss_foc",
                    "implied_value", "implied_value_gap_to_pfi", "benchmark_policy_mae",
                ],
            ),
            "calibration": full_cal[name],
            "budget_sec": FRICTIONAL_BUDGET_SEC,
            "snapshot_targets": SNAPSHOT_TARGETS,
            "decay_rate": full_cal[name]["derived_decay"],
            "n_anneal": full_cal[name]["n_anneal"],
        }
        entry["selected"] = select_best_checkpoint(bundle_name, entry, "benchmark_policy_mae")
        persist_method(bundle_name, entry)
    entry["color"] = FRICTIONLESS_VARIANTS[name]["color"]
    entry["name"] = name
    full_results[name] = entry

full_rows = []
for name in FRICTIONAL_VARIANTS:
    entry = full_results[name]
    sel = entry["selected"]
    idx = entry["checkpoint_index"][sel["selected_step"]]
    metrics = entry["checkpoint_metrics"]
    row = {
        "variant": entry["label"],
        "selected_step": int(sel["selected_step"]),
        "selected_time_sec": float(sel["selected_time"]),
        "benchmark_policy_mae": float(metrics["benchmark_policy_mae"][idx]),
        "implied_value": float(metrics["implied_value"][idx]),
        "implied_value_gap_to_pfi": float(metrics["implied_value_gap_to_pfi"][idx]),
        "loss_br": float(metrics["loss_br"][idx]),
        "loss_foc": float(metrics["loss_foc"][idx]),
        "wall_time_sec": float(entry["result"]["wall_time_sec"]),
    }
    full_rows.append(row)

save_summary_rows(RUN, full_rows, filename="full_summary.csv")
print(f"{'Variant':>18s} | {'step':>6s} | {'time':>8s} | {'Policy MAE':>12s} | {'Value':>12s} | {'Gap to PFI':>12s}")
print("-" * 92)
for row in full_rows:
    print(
        f"{row['variant']:>18s} | {row['selected_step']:6d} | {row['selected_time_sec']:8.1f} | "
        f"{row['benchmark_policy_mae']:12.4f} | {row['implied_value']:12.4f} | {row['implied_value_gap_to_pfi']:12.4f}"
    )


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
metric_panels = [
    ("benchmark_policy_mae", "Benchmark Policy MAE", None),
    ("implied_value", "Implied Value", pfi_value),
    ("loss_br", "Bellman Residual Loss", None),
    ("loss_foc", "FOC Loss", None),
]
for ax, (metric_key, title, ref_line) in zip(axes.ravel(), metric_panels):
    for name in FRICTIONAL_VARIANTS:
        entry = full_results[name]
        m = entry["checkpoint_metrics"]
        ax.plot(m["elapsed_sec"], m[metric_key], color=entry["color"], lw=1.6, label=entry["label"])
        add_selected_marker(ax, entry, metric_key)
    if ref_line is not None:
        ax.axhline(ref_line, color="black", lw=1.2, ls="--", alpha=0.8)
    ax.set_title(title, fontweight="bold")
    ax.set_xlabel("Elapsed seconds")
    ax.grid(True, alpha=0.3)
handles = [Line2D([0], [0], color=full_results[name]["color"], lw=2, label=full_results[name]["label"]) for name in FRICTIONAL_VARIANTS]
handles += [Line2D([0], [0], color="black", lw=1.2, ls="--", label="PFI")]
fig.legend(handles=handles, loc="lower center", bbox_to_anchor=(0.5, -0.03), ncol=3, frameon=False)
fig.suptitle("Frictional BRM Comparison — External and Internal Metrics", fontweight="bold", fontsize=13)
plt.tight_layout(rect=[0, 0.06, 1, 0.96])
persist_figure(fig, "full_convergence.png")
plt.show()

pfi_kprime_z = np.asarray(pfi_kprime_interp(np.column_stack([z_grid, np.full_like(z_grid, k_star)]))).reshape(-1)
pfi_kprime_k = np.asarray(pfi_kprime_interp(np.column_stack([np.full_like(k_grid, z_mean), k_grid]))).reshape(-1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].plot(z_grid, pfi_kprime_z, color="black", lw=2, ls="--", label="PFI")
for name in FRICTIONAL_VARIANTS:
    entry = full_results[name]
    restore_entry_at_step(entry, entry["selected"]["selected_step"])
    kp = get_policy_kprime(entry["policy"], env_full_eval, s_at_kstar)
    axes[0].plot(z_grid, kp, color=entry["color"], lw=1.6, label=entry["label"])
axes[0].set_title("k'(z) at k = k*", fontweight="bold")
axes[0].set_xlabel("Productivity z")
axes[0].set_ylabel("Next capital k'")
axes[0].grid(True, alpha=0.3)

axes[1].plot(k_grid, pfi_kprime_k, color="black", lw=2, ls="--", label="PFI")
for name in FRICTIONAL_VARIANTS:
    entry = full_results[name]
    kp = get_policy_kprime(entry["policy"], env_full_eval, s_at_zmean)
    axes[1].plot(k_grid, kp, color=entry["color"], lw=1.6, label=entry["label"])
axes[1].plot(k_grid, k_grid, color="gray", lw=1.0, ls=":", alpha=0.6)
axes[1].set_title("k'(k) at z = E[z]", fontweight="bold")
axes[1].set_xlabel("Current capital k")
axes[1].set_ylabel("Next capital k'")
axes[1].grid(True, alpha=0.3)

fig.legend(handles=handles, loc="lower center", bbox_to_anchor=(0.5, -0.02), ncol=3, frameon=False)
fig.suptitle("Frictional Selected Checkpoints — Policy Slice Validation", fontweight="bold", fontsize=13)
plt.tight_layout(rect=[0, 0.08, 1, 0.95])
persist_figure(fig, "full_slices.png")
plt.show()


---
# Section 6: Additional Robustness Test A — Seed Sweep

This section is a robustness test, not part of the main headline comparison.
The dataset is held fixed. The seed changes the training-side randomness and network initialization.


In [ ]:
seed_sweep_rows = []
if not RUN_SEED_SWEEP:
    print("Seed sweep disabled for this profile.")
else:
    seed_variants = ["joint_warm", "separated_warm"]
    for variant in seed_variants:
        spec = FRICTIONLESS_VARIANTS[variant]
        bundle_prefix = f"seed_{variant}"
        cal = frictionless_cal[variant]
        for seed in SEED_SWEEP_VALUES:
            bundle_name = f"{bundle_prefix}_{seed}"
            if RUN["mode"] == "load":
                entry = load_saved_method(bundle_name, env_frictionless_eval)
            else:
                run_seed = (int(seed), MASTER_SEED[1])
                seed_runtime(run_seed, "seed_sweep", variant, strict_reproducibility=STRICT_REPRODUCIBILITY)
                checkpoint_history = []
                policy = make_policy(env_frictionless_eval, name=f"policy_{bundle_name}", seed=fold_in_seed(run_seed, "policy"))
                value_net = make_value_net(env_frictionless_eval, name=f"value_{bundle_name}", seed=fold_in_seed(run_seed, "value"))
                config = sanitize_config(
                    replace(frictionless_base, master_seed=run_seed),
                    n_steps=cal["n_step_cap"],
                    eval_interval=cal["eval_interval"],
                    max_wall_time_sec=FRICTIONLESS_BUDGET_SEC,
                    checkpoint_history=checkpoint_history,
                    snapshot_targets=SNAPSHOT_TARGETS,
                    warm_start_steps=spec["warm_start_steps"],
                )
                result = spec["trainer"](
                    env_frictionless_eval,
                    policy,
                    value_net,
                    frictionless_train_flat,
                    config=config,
                    eval_callback=frictionless_eval_callback,
                )
                entry = {
                    "label": spec["label"],
                    "section": "seed_sweep",
                    "policy": result["policy"],
                    "value_net": result["value_net"],
                    "result": result,
                    "checkpoint_history": checkpoint_history,
                    "checkpoint_metrics": extract_checkpoint_metrics(
                        result,
                        ["step", "elapsed_sec", "train_temperature", "loss", "loss_br", "loss_foc", "analytic_policy_mae", "implied_value", "euler_residual"],
                    ),
                    "calibration": cal,
                    "budget_sec": FRICTIONLESS_BUDGET_SEC,
                    "snapshot_targets": SNAPSHOT_TARGETS,
                }
                entry["selected"] = select_best_checkpoint(bundle_name, entry, "analytic_policy_mae")
                persist_method(bundle_name, entry)
            idx = entry["checkpoint_index"][entry["selected"]["selected_step"]]
            seed_sweep_rows.append({
                "variant": spec["label"],
                "seed": int(seed),
                "analytic_policy_mae": float(entry["checkpoint_metrics"]["analytic_policy_mae"][idx]),
                "implied_value": float(entry["checkpoint_metrics"]["implied_value"][idx]),
            })
    save_summary_rows(RUN, seed_sweep_rows, filename="seed_sweep_summary.csv")
    print(seed_sweep_rows[:min(len(seed_sweep_rows), 10)])

    fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.5))
    x_map = {"Joint + Warm": 0, "Separated + Warm": 1}
    for row in seed_sweep_rows:
        x = x_map[row["variant"]]
        jitter = 0.08 * (0.5 - np.random.RandomState(row["seed"]).rand())
        color = FRICTIONLESS_VARIANTS["joint_warm"]["color"] if row["variant"] == "Joint + Warm" else FRICTIONLESS_VARIANTS["separated_warm"]["color"]
        axes[0].scatter(x + jitter, row["analytic_policy_mae"], color=color, s=45, alpha=0.8)
        axes[1].scatter(x + jitter, row["implied_value"], color=color, s=45, alpha=0.8)
    for ax, title in zip(axes, ["Analytical Policy MAE", "Implied Value"]):
        ax.set_xticks([0, 1], ["Joint + Warm", "Separated + Warm"])
        ax.set_title(title, fontweight="bold")
        ax.grid(True, axis="y", alpha=0.3)
    fig.suptitle("Seed Sweep Robustness (Frictionless)", fontweight="bold", fontsize=13)
    plt.tight_layout()
    persist_figure(fig, "seed_sweep.png")
    plt.show()


---
# Section 7: Additional Robustness Test B — Warm-Start Strength Sweep

This section varies **warm-start strength** through the number of warm-start critic steps.
It is designed to test how sensitive BRM is to the early critic gradient field.


In [ ]:
warm_sweep_rows = []
if not RUN_WARM_SWEEP:
    print("Warm-start sweep disabled for this profile.")
else:
    warm_variants = ["joint", "separated"]
    for family in warm_variants:
        trainer = train_brm_joint if family == "joint" else train_brm
        color = "#C44E52" if family == "joint" else "#4C72B0"
        cal = frictionless_cal[f"{family}_warm"]
        for label, warm_steps in WARM_SWEEP_LEVELS:
            bundle_name = f"warm_{family}_{label}"
            if RUN["mode"] == "load":
                entry = load_saved_method(bundle_name, env_frictionless_eval)
            else:
                seed_experiment("warm_sweep", family, label)
                checkpoint_history = []
                policy = make_policy(env_frictionless_eval, name=f"policy_{bundle_name}", seed=derived_seed("warm_sweep", family, "policy"))
                value_net = make_value_net(env_frictionless_eval, name=f"value_{bundle_name}", seed=derived_seed("warm_sweep", family, "value"))
                config = sanitize_config(
                    frictionless_base,
                    n_steps=cal["n_step_cap"],
                    eval_interval=cal["eval_interval"],
                    max_wall_time_sec=FRICTIONLESS_BUDGET_SEC,
                    checkpoint_history=checkpoint_history,
                    snapshot_targets=SNAPSHOT_TARGETS,
                    warm_start_steps=int(warm_steps),
                )
                result = trainer(
                    env_frictionless_eval,
                    policy,
                    value_net,
                    frictionless_train_flat,
                    config=config,
                    eval_callback=frictionless_eval_callback,
                )
                entry = {
                    "label": f"{family.title()} {label}",
                    "section": "warm_sweep",
                    "policy": result["policy"],
                    "value_net": result["value_net"],
                    "result": result,
                    "checkpoint_history": checkpoint_history,
                    "checkpoint_metrics": extract_checkpoint_metrics(
                        result,
                        ["step", "elapsed_sec", "train_temperature", "loss", "loss_br", "loss_foc", "analytic_policy_mae", "implied_value", "euler_residual"],
                    ),
                    "calibration": cal,
                    "budget_sec": FRICTIONLESS_BUDGET_SEC,
                    "snapshot_targets": SNAPSHOT_TARGETS,
                }
                entry["selected"] = select_best_checkpoint(bundle_name, entry, "analytic_policy_mae")
                persist_method(bundle_name, entry)
            idx = entry["checkpoint_index"][entry["selected"]["selected_step"]]
            warm_sweep_rows.append({
                "family": family.title(),
                "warm_label": label,
                "warm_steps": int(warm_steps),
                "analytic_policy_mae": float(entry["checkpoint_metrics"]["analytic_policy_mae"][idx]),
                "implied_value": float(entry["checkpoint_metrics"]["implied_value"][idx]),
                "color": color,
            })
    save_summary_rows(RUN, warm_sweep_rows, filename="warm_sweep_summary.csv")

    level_positions = {label: i for i, (label, _) in enumerate(WARM_SWEEP_LEVELS)}
    fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.5))
    for family in ["Joint", "Separated"]:
        rows = [r for r in warm_sweep_rows if r["family"] == family]
        rows = sorted(rows, key=lambda r: level_positions[r["warm_label"]])
        xs = [r["warm_label"] for r in rows]
        axes[0].plot(xs, [r["analytic_policy_mae"] for r in rows], marker="o", lw=1.8, color=rows[0]["color"], label=family)
        axes[1].plot(xs, [r["implied_value"] for r in rows], marker="o", lw=1.8, color=rows[0]["color"], label=family)
    axes[0].set_title("Analytical Policy MAE", fontweight="bold")
    axes[1].set_title("Implied Value", fontweight="bold")
    for ax in axes:
        ax.grid(True, alpha=0.3)
    fig.legend(loc="lower center", bbox_to_anchor=(0.5, -0.02), ncol=2, frameon=False)
    fig.suptitle("Warm-Start Strength Sweep (Frictionless)", fontweight="bold", fontsize=13)
    plt.tight_layout(rect=[0, 0.08, 1, 1])
    persist_figure(fig, "warm_sweep.png")
    plt.show()


---
# Section 8: Conclusions

Target interpretation for this notebook:
- **Joint-update BRM** should look more fragile than the refined separated-update BRM.
- **Warm start** should materially help BRM in the frictionless benchmark.
- In the **frictional setting**, even refined BRM can train internally while external solution quality remains poor.
- The robustness sections should help distinguish a structural instability from one lucky or unlucky run.
